In [5]:
import pandas as pd

df = pd.read_csv("medical_insurance.csv")

print(df.head())

   age     sex     bmi  children smoker     region      charges
0   19  female  27.900         0    yes  southwest  16884.92400
1   18    male  33.770         1     no  southeast   1725.55230
2   28    male  33.000         3     no  southeast   4449.46200
3   33    male  22.705         0     no  northwest  21984.47061
4   32    male  28.880         0     no  northwest   3866.85520


In [6]:
df_encoded = pd.get_dummies(
    df,
    columns=['sex', 'smoker', 'region'],
    drop_first=True
)

print(df_encoded.head())

   age     bmi  children      charges  sex_male  smoker_yes  region_northwest  \
0   19  27.900         0  16884.92400     False        True             False   
1   18  33.770         1   1725.55230      True       False             False   
2   28  33.000         3   4449.46200      True       False             False   
3   33  22.705         0  21984.47061      True       False              True   
4   32  28.880         0   3866.85520      True       False              True   

   region_southeast  region_southwest  
0             False              True  
1              True             False  
2              True             False  
3             False             False  
4             False             False  


In [7]:
X = df_encoded.drop("charges", axis=1)

y = df_encoded["charges"]

print(X.shape)
print(y.shape)

(2772, 8)
(2772,)


In [8]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print(X_train.shape)
print(X_test.shape)

(2217, 8)
(555, 8)


In [9]:
from sklearn.ensemble import RandomForestRegressor

model = RandomForestRegressor(
    n_estimators=100,
    random_state=42
)

model.fit(X_train, y_train)

print("Model Trained Successfully")

Model Trained Successfully


In [10]:
predictions = model.predict(X_test)

print(predictions[:5])

[ 9644.9264553 28276.2598945 11953.96374    2040.758722   3586.5431711]


In [11]:
from sklearn.metrics import r2_score, mean_absolute_error

r2 = r2_score(y_test, predictions)
mae = mean_absolute_error(y_test, predictions)

print("R2 Score:", r2)
print("MAE:", mae)

R2 Score: 0.950722902871192
MAE: 1282.1300664536232


In [12]:
import pickle

pickle.dump(
    model,
    open("insurance_model.pkl", "wb")
)

print("Model Saved Successfully")

Model Saved Successfully


In [13]:
pickle.dump(
    X.columns,
    open("model_columns.pkl", "wb")
)

print(X.columns)

Index(['age', 'bmi', 'children', 'sex_male', 'smoker_yes', 'region_northwest',
       'region_southeast', 'region_southwest'],
      dtype='str')


In [15]:
import pickle
import pandas as pd

model = pickle.load(open("insurance_model.pkl", "rb"))

sample = pd.DataFrame({
    "age":[25],
    "bmi":[24.5],
    "children":[1],
    "sex_male":[1],
    "smoker_yes":[0],
    "region_northwest":[0],
    "region_southeast":[1],
    "region_southwest":[0]
})

prediction = model.predict(sample)

print(prediction)

[7528.7667053]


In [18]:
%%writefile insurance_gui.py
import tkinter as tk
from tkinter import ttk, messagebox
import pandas as pd
import pickle

# Load Model
model = pickle.load(open("insurance_model.pkl", "rb"))

# Prediction Function
def predict_insurance():
    try:
        age = int(entry_age.get())
        bmi = float(entry_bmi.get())
        children = int(entry_children.get())

        gender = gender_var.get()
        smoker = smoker_var.get()
        region = region_var.get()

        # Convert categorical values
        sex_male = 1 if gender == "Male" else 0
        smoker_yes = 1 if smoker == "Yes" else 0

        region_northwest = 1 if region == "Northwest" else 0
        region_southeast = 1 if region == "Southeast" else 0
        region_southwest = 1 if region == "Southwest" else 0

        # Create DataFrame
        new_data = pd.DataFrame({
            "age": [age],
            "bmi": [bmi],
            "children": [children],
            "sex_male": [sex_male],
            "smoker_yes": [smoker_yes],
            "region_northwest": [region_northwest],
            "region_southeast": [region_southeast],
            "region_southwest": [region_southwest]
        })

        prediction = model.predict(new_data)

        result_label.config(
            text=f"Predicted Insurance Cost: ₹{prediction[0]:,.2f}"
        )

    except Exception as e:
        messagebox.showerror("Error", str(e))


# Main Window
root = tk.Tk()
root.title("Medical Insurance Cost Prediction")
root.geometry("600x700")
root.configure(bg="#f5f5f5")

# Title
title = tk.Label(
    root,
    text="🏥 Medical Insurance Cost Prediction",
    font=("Arial", 20, "bold"),
    bg="#f5f5f5",
    fg="darkblue"
)
title.pack(pady=20)

# Frame
frame = tk.Frame(root, bg="white", bd=2, relief="ridge")
frame.pack(padx=20, pady=10, fill="both")

# Age
tk.Label(frame, text="Age", bg="white",
         font=("Arial", 12)).pack(pady=5)

entry_age = tk.Entry(frame, font=("Arial", 12))
entry_age.pack(pady=5)

# BMI
tk.Label(frame, text="BMI", bg="white",
         font=("Arial", 12)).pack(pady=5)

entry_bmi = tk.Entry(frame, font=("Arial", 12))
entry_bmi.pack(pady=5)

# Children
tk.Label(frame, text="Children", bg="white",
         font=("Arial", 12)).pack(pady=5)

entry_children = tk.Entry(frame, font=("Arial", 12))
entry_children.pack(pady=5)

# Gender Dropdown
tk.Label(frame, text="Gender", bg="white",
         font=("Arial", 12)).pack(pady=5)

gender_var = tk.StringVar()
gender_dropdown = ttk.Combobox(
    frame,
    textvariable=gender_var,
    values=["Male", "Female"],
    state="readonly"
)
gender_dropdown.pack(pady=5)
gender_dropdown.current(0)

# Smoker Dropdown
tk.Label(frame, text="Smoker", bg="white",
         font=("Arial", 12)).pack(pady=5)

smoker_var = tk.StringVar()
smoker_dropdown = ttk.Combobox(
    frame,
    textvariable=smoker_var,
    values=["Yes", "No"],
    state="readonly"
)
smoker_dropdown.pack(pady=5)
smoker_dropdown.current(1)

# Region Dropdown
tk.Label(frame, text="Region", bg="white",
         font=("Arial", 12)).pack(pady=5)

region_var = tk.StringVar()
region_dropdown = ttk.Combobox(
    frame,
    textvariable=region_var,
    values=[
        "Northeast",
        "Northwest",
        "Southeast",
        "Southwest"
    ],
    state="readonly"
)
region_dropdown.pack(pady=5)
region_dropdown.current(0)

# Predict Button
predict_btn = tk.Button(
    root,
    text="Predict Insurance Cost",
    command=predict_insurance,
    bg="green",
    fg="white",
    font=("Arial", 14, "bold"),
    width=22
)
predict_btn.pack(pady=20)

# Result Label
result_label = tk.Label(
    root,
    text="",
    font=("Arial", 16, "bold"),
    bg="#f5f5f5",
    fg="red"
)
result_label.pack(pady=20)

# Footer
footer = tk.Label(
    root,
    text="Developed by Your Name",
    font=("Arial", 10),
    bg="#f5f5f5"
)
footer.pack(side="bottom", pady=10)

root.mainloop()

Writing insurance_gui.py


In [ ]:
!python insurance_gui.py